1. Instalación

In [ ]:
# Instalar dependencias (ejecutar solo la primera vez)
%pip install biopython localcider matplotlib torch

2. Imports

In [2]:
import os
import csv
import zipfile
import shutil

from itertools import combinations

from Bio import AlignIO
from localcider.sequenceParameters import SequenceParameters
from pathlib import Path

import matplotlib.pyplot as plt


Confirmación del directorio y creación de carpetas

In [ ]:


NOTEBOOK_DIR = Path.cwd()

print("Directorio actual:", NOTEBOOK_DIR)

os.makedirs("../results", exist_ok=True)
os.makedirs("../results/csv", exist_ok=True)
os.makedirs("../results/figures", exist_ok=True)



3. Cargar MSA

In [ ]:
E1A_FILE = "../data/E1A_MSA.fasta"

alineamiento = AlignIO.read(
    open(E1A_FILE),
    "fasta"
)

print("Cantidad de secuencias:", len(alineamiento))

4.Funciones

Identidad()


In [ ]:
# ==================================================
# FUNCIONES DE ANÁLISIS DE SECUENCIAS
# ==================================================


def identidad(seq1, seq2):
    """
    Calcula el porcentaje de identidad entre dos secuencias
    ignorando las posiciones con gaps.
    """

    matcheo = 0
    mismacheo = 0

    longitud = len(seq1)

    for pos in range(longitud):

        if seq1[pos] != "-" and seq2[pos] != "-":

            if seq1[pos] == seq2[pos]:
                matcheo += 1
            else:
                mismacheo += 1

    return matcheo / (matcheo + mismacheo)

contar_aminoacidos()

In [ ]:
def contar_aminoacidos(seq):
    """
    Cuenta la frecuencia de cada aminoácido presente
    en una secuencia.
    """

    conteo = {}

    for aa in seq:

        if aa in conteo:
            conteo[aa] += 1
        else:
            conteo[aa] = 1

    return conteo

contar_aminoacidos_totales()

In [ ]:
def contar_aminoacidos_totales(alineamiento):

    conteo = {}

    for i in range(len(alineamiento)):

        for aa in alineamiento[i].seq:

            if aa in conteo:
                conteo[aa] += 1
            else:
                conteo[aa] = 1

    return conteo

calcular_cargas()

In [ ]:
def calcular_cargas(secuencia):
    """
    Calcula la cantidad de aminoácidos con carga positiva,
    negativa y neutra presentes en la secuencia.
    """
    cargas = {
        '+': 0,
        '-': 0,
        'neutro': 0
    }

    aminoacidos_cargados = {
        'R': '+',
        'K': '+',
        'D': '-',
        'E': '-',
        'H': '+',
        'C': 'neutro',
        'Y': 'neutro'
    }

    for aminoacido in secuencia:

        if aminoacido in aminoacidos_cargados:

            carga = aminoacidos_cargados[aminoacido]

            cargas[carga] += 1

    return cargas

carga_neta()

In [ ]:
def carga_neta(secuencia):
    """
    Calcula la carga neta de una secuencia utilizando
    los aminoácidos cargados R, K, D y E.
    """
    carga = 0

    aminoacidos_cargados = {
        'R': 1,
        'K': 1,
        'D': -1,
        'E': -1
    }

    for aminoacido in secuencia:

        if aminoacido in aminoacidos_cargados:

            carga += aminoacidos_cargados[aminoacido]

    return carga

radio()

In [ ]:
def radio(secuencia):
    """
    Estima el radio hidrodinámico de una proteína a partir
    de su longitud, contenido de prolina y carga neta.
    """
    prol = contar_aminoacidos(secuencia).get("P", 0)

    carga = carga_neta(secuencia)

    longitud = len(
        secuencia.replace("-", "")
    )

    if longitud == 0:
        return 0

    rh = (
        (1.24 * prol / longitud + 0.904)
        * (0.00759 * abs(carga) + 0.963)
        * 2.49
        * 0.901
        * longitud ** 0.509
    )

    return rh

kappa()

In [ ]:
def kappa(secuencia):
    """
    Calcula el parámetro kappa utilizando LocalCIDER,
    que describe la distribución de cargas en la secuencia.
    """
    seq = SequenceParameters(
        str(secuencia).replace("-", "")
    )

    return seq.get_kappa()

resultadosPrimarios()

In [ ]:
def resultadosPrimarios(alineamiento):
    """
    Genera un archivo CSV con longitud, radio hidrodinámico
    y valor de kappa para cada proteína del alineamiento.
    """
    documento = open(
        "../results/csv/primary_results.csv",
        "w"
    )

    documento.write(
    "Proteina;Longitud;Radio_Hidrodinamico;Kappa\n"
    )

    for seq in alineamiento:

        nombre = seq.id.split("|")[3]

        l = str(
            len(seq.seq.replace("-", ""))
        )

        r = str(radio(seq.seq))

        k = str(kappa(seq.seq))

        texto = (
            nombre + ";"
            + l + ";"
            + r + ";"
            + k + "\n"
        )

        documento.write(texto)

    documento.close()

5. Longitud promedio

In [ ]:
# ==================================================
# CÁLCULO DE ESTADÍSTICAS BÁSICAS
# ==================================================


suma = 0

for registro in alineamiento:

    longitud = len(
        registro.seq.replace("-", "")
    )

    print(
        registro.id.split("|")[3],
        longitud
    )

    suma += longitud

print(
    "Longitud promedio:",
    suma / len(alineamiento)
)

6. Divergencia / identidad

In [ ]:
for reg in combinations(alineamiento, 2):

    print(
        reg[0].id.split("|")[3],
        reg[1].id.split("|")[3],
        identidad(
            reg[0].seq,
            reg[1].seq
        )
    )

7. Conteo de aminoácidos

In [ ]:
conteo_total = contar_aminoacidos_totales(
    alineamiento
)

print(conteo_total)

Guardar

In [ ]:
with open(
    "../results/csv/average_amino_acid_composition.csv",
    "w"
) as documento:

    documento.write(
        str(conteo_total)
    )

8. Radio y Kappa

In [ ]:
resultadosPrimarios(alineamiento)

9. Gráficos

In [ ]:
# ==================================================
# GENERACIÓN DE GRÁFICOS
# ==================================================

radioGrafico = []
kappaGrafico = []
longitudGrafico = []

for secuencia in alineamiento:

    radioGrafico.append(
        radio(secuencia.seq)
    )

    kappaGrafico.append(
        kappa(secuencia.seq)
    )

    longitudGrafico.append(
        len(secuencia.seq.replace("-", ""))
    )

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    kappaGrafico,
    radioGrafico,
    s=5
)

plt.xlabel("Kappa")
plt.ylabel("Radio Hidrodinámico")

plt.savefig(
    "../results/figures/kappa_vs_radius.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Longitud vs Radio

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    longitudGrafico,
    radioGrafico,
    s=5
)

plt.xlabel("Longitud")
plt.ylabel("Radio Hidrodinámico")

plt.savefig(
    "../results/figures/length_vs_radius.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

10 - AIUPred

Archivos de entrada

In [ ]:
# ==================================================
# ANÁLISIS DE DESORDEN CON AIUPRED
# ==================================================


E1A_MSA1 = "../data/E1A_MSA1.fasta"
E1A_MSA2 = "../data/E1A_MSA2.fasta"

E1A_MSA_sin_gaps1 = "../data/E1A_MSA_sin_gaps1.fasta"
E1A_MSA_sin_gaps2 = "../data/E1A_MSA_sin_gaps2.fasta"

Ejecutar AIUPred

In [ ]:
# Extraer AIUPred (solo la primera vez)
os.makedirs("./aiupred", exist_ok=True)

with zipfile.ZipFile("../data/AIUPred-1.0.zip", "r") as zip_ref:
    zip_ref.extractall("./aiupred")

print("AIUPred extraído correctamente")

AIUPred extraído correctamente


In [ ]:
os.makedirs("../results", exist_ok=True)

"""
Ejecuta AIUPred sobre los archivos FASTA preparados
y guarda las predicciones de desorden en archivos TXT.
"""

# Ejecutar AIUPred
!python ./aiupred/AIUPred-1.0/aiupred.py -i $E1A_MSA_sin_gaps1 -o ../results/resultadosE1A_sin_gaps1.txt

!python ./aiupred/AIUPred-1.0/aiupred.py -i $E1A_MSA_sin_gaps2 -o ../results/resultadosE1A_sin_gaps2.txt

!python ./aiupred/AIUPred-1.0/aiupred.py -i $E1A_MSA1 -o ../results/resultadosE1A_con_gaps1.txt

!python ./aiupred/AIUPred-1.0/aiupred.py -i $E1A_MSA2 -o ../results/resultadosE1A_con_gaps2.txt

Combinar resultados

In [ ]:
"""
Une los resultados parciales generados por AIUPred
en archivos únicos para su posterior análisis.
"""

with open("../results/resultadosE1A_sin_gaps.txt", "wb") as salida:

    for archivo in [
        "../results/resultadosE1A_sin_gaps1.txt",
        "../results/resultadosE1A_sin_gaps2.txt"
    ]:

        with open(archivo, "rb") as entrada:
            shutil.copyfileobj(entrada, salida)

with open("../results/resultadosE1A_con_gaps.txt", "wb") as salida:

    for archivo in [
        "../results/resultadosE1A_con_gaps1.txt",
        "../results/resultadosE1A_con_gaps2.txt"
    ]:

        with open(archivo, "rb") as entrada:
            shutil.copyfileobj(entrada, salida)

Convertir salida AIUPred a CSV

In [ ]:
"""
Convierte la salida de AIUPred a un formato tabular
que puede integrarse con el alineamiento original.
"""


E1A_AIUPRED = "../results/resultadosE1A_sin_gaps.txt"

ALINEAMIENTO_DESORDEN = "../results/csv/disorder_alignment.csv"

with open(ALINEAMIENTO_DESORDEN, "w") as grabar:

    with open(E1A_AIUPRED, "r") as archivo:

        for linea in archivo:

            if linea:

                if linea[0] != "\n" and linea[0] != "#":

                    if linea[0] == ">":

                        grabar.write(
                            "\n" +
                            linea.split("|")[3]
                        )

                    else:

                        grabar.write(
                            ";" +
                            linea.split()[2]
                        )

Reconstruir gaps

In [ ]:
ALINEAMIENTO_DESORDEN_GAPS = \
    "../results/csv/disorder_alignment_with_gaps.csv"

with open(
    ALINEAMIENTO_DESORDEN,
    "r",
    newline=""
) as archivo:

    lector = csv.reader(
        archivo,
        delimiter=";"
    )

    next(lector)

    lineas = list(lector)

with open(
    ALINEAMIENTO_DESORDEN_GAPS,
    "w"
) as grabar:

    j = 0

    for secu in alineamiento:

        grabar.write(
            "\n" +
            secu.id.split("|")[3]
        )

        caminador = 1

        for pos in secu.seq:

            if pos != "-":

                grabar.write(
                    ";" +
                    lineas[j][caminador]
                )

                caminador += 1

            else:

                grabar.write(";")

        j += 1